# Week 6, Day 1: Curate a memorabilia price dataset

**Lesson:** [week6/day1.ipynb](../../../week6/day1.ipynb), *Data curation*. **About 25 minutes.**

**What you practice here**

- Removing duplicates the lesson's way, after a seeded shuffle: repeated titles first, then repeated full texts.
- Rebalancing a skewed dataset with weights: prices scaled to 0-1 and squared, damped for an over-represented category, then normalized to sum to 1.
- Drawing a reproducible weighted sample without replacement with numpy's `random.seed` and `random.choice`.
- Splitting the curated sample into train, validation and test sets, as the lesson does before it pushes them to the Hub.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week6/day1", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson curates its Amazon products before any model sees them. It removes duplicates, then draws a weighted
sample so that cheap items and an over-represented category do not crowd out the rest, then splits the sample into
train, validation and test sets. Do the same for a marketplace's wrestling memorabilia listings (programs, posters,
action figures, title belts, autographs and fan gear), where cheap action figures fill most of the pages.

Each listing is an object with the attributes of the lesson's `Item`: `title`, `full` (its text), `price` and
`category`. The checks use `SimpleNamespace` listings, and in Try it for real the same functions run on the lesson's
own items. numpy is the only library you need.

## What each call should give

```python
# four listings whose (title, full) are ("A", "a"), ("A", "b"), ("B", "c") and ("C", "c")
[x.full for x in dedupe(listings)]
# ['a', 'c']    the second "A" repeats a title, and "C" repeats the text of "B"

# prices 10, 110, 210 and 410; the first and the last are action figures
sampling_weights(items, {"Action figures": 0.05})
# array([0.    , 0.1724, 0.6897, 0.1379])

draw_sample(items, weights, 2)
# two different items, drawn by weight as the lesson draws them, with the seed 42

split(list(range(10)), 6, 3)
# ([0, 1, 2, 3, 4, 5], [6, 7, 8], [9])
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import numpy as np

### 1 · `dedupe(items)`

Drops repeated listings as the lesson does after its shuffle: repeated titles first, then repeated full texts.
Return a new list that keeps only the first item with each `title`, then, among the items kept, only the first with
each `full` text, in their original order. An item dropped for its title does not count when the full texts are
compared, as in the lesson's two passes. Do not change the list you are given.

<details><summary>Hint</summary>

The lesson's idiom for one pass keeps a set of what it has seen (`seen.add` returns `None`, so the test reads as
"seen before"). Do it once with `x.title`, then again on what is left with `x.full`.

```python
seen = set()
kept = [x for x in items if not (x.title in seen or seen.add(x.title))]
```

</details>

In [ ]:
def dedupe(items):
    # Write dedupe here, as described above.
    raise NotImplementedError("dedupe is not written yet")

In [ ]:
check("dedupe")

### 2 · `sampling_weights(items, damping)`

Gives each item its weight for the draw, as the lesson does before its random choice, and returns the weights as a
numpy array that sums to 1. Scale each price to the range 0 to 1, from the lowest price to the highest (the lesson
adds a tiny 1e-9 to the range), and square it. Then multiply the weights of each category in `damping` by its
factor: `damping` maps a category to a number, such as `{"Action figures": 0.05}`, and the other categories keep a
factor of 1. Last, divide every weight by their total. The cheapest item gets 0, as in the lesson. You can assume at
least two different prices.

<details><summary>Hint</summary>

With the prices and the categories in numpy arrays, the lesson's formula is one line, and a mask picks out one
category's weights:

```python
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)
w = p ** 2
w[categories == "Action figures"] *= 0.05
return w / w.sum()
```

</details>

In [ ]:
def sampling_weights(items, damping):
    # Write sampling_weights here, as described above.
    raise NotImplementedError("sampling_weights is not written yet")

In [ ]:
check("sampling_weights")

### 3 · `draw_sample(items, weights, size, seed=42)`

Draws a reproducible weighted sample of different items with the lesson's two calls. Seed numpy's global random
generator with `seed`, then have numpy choose `size` different positions out of `len(items)`, without replacement and
with the probabilities `weights`. Return the items at those positions as a list, in the order drawn. The checks
compare your sample with the lesson's two calls, so use them rather than numpy's newer `default_rng`, which draws
differently. Do not change `items`. You can assume at least `size` weights are above 0.

<details><summary>Hint</summary>

```python
np.random.seed(seed)
idx = np.random.choice(len(items), size=size, replace=False, p=weights)
```

</details>

In [ ]:
def draw_sample(items, weights, size, seed=42):
    # Write draw_sample here, as described above.
    raise NotImplementedError("draw_sample is not written yet")

In [ ]:
check("draw_sample")

### 4 · `split(sample, train_size, val_size)`

Splits a sample into train, validation and test sets, as the lesson does with slices of its 820,000 items. Return a
tuple `(train, validation, test)` of three lists: the first `train_size` items, the next `val_size` items, and all
the rest, in order.

<details><summary>Hint</summary>

The lesson's slices: `sample[:800_000]`, `sample[800_000:810_000]` and `sample[810_000:]`.

</details>

In [ ]:
def split(sample, train_size, val_size):
    # Write split here, as described above.
    raise NotImplementedError("split is not written yet")

In [ ]:
check("split")

## Check everything

The cell below runs all 7 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 6, Day 1 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Nothing here calls a model, so you need no key. Loading the two categories takes a few minutes, as in the lesson, and needs the course environment (`datasets` and `matplotlib`).

1. Load the lesson's raw `Appliances` rows with `load_dataset("McAuley-Lab/Amazon-Reviews-2023",
   "raw_meta_Appliances", split="full", trust_remote_code=True)`, and find the most expensive item yourself, reading
   each price with `float()` inside `try`/`except ValueError`, as the lesson does.
2. Put the course's `week6` folder on the path with `sys.path.insert(0, str(ROOT / "week6"))`, and load the lesson's
   `Appliances` and `Musical_Instruments` items with its `ItemLoader`.
3. Shuffle them with `random.seed(42)`, run `dedupe`, and count the categories with `Counter`. Then draw 20,000 items
   with `sampling_weights` (damp the larger category) and `draw_sample`.
4. Plot the price histogram of all the items and of the sample, as the lesson does, then `split` the sample and
   print the sizes of the three sets.

<details><summary>Stuck? One way to write it</summary>

```python
import random
import sys
from collections import Counter

import matplotlib.pyplot as plt
from datasets import load_dataset

rows = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Appliances", split="full", trust_remote_code=True)
top, top_price = None, 0.0
for row in rows:
    try:
        price = float(row["price"])
    except ValueError:
        continue
    if price > top_price:
        top, top_price = row, price
print(f"The most expensive appliance is {top['title']} at ${top_price:,.2f}")

sys.path.insert(0, str(ROOT / "week6"))  # the lesson's pricer package
from pricer.loaders import ItemLoader

items = ItemLoader("Appliances").load() + ItemLoader("Musical_Instruments").load()
random.seed(42)
random.shuffle(items)
items = dedupe(items)
print(Counter(item.category for item in items))

weights = sampling_weights(items, {"Musical_Instruments": 0.5})  # damp whichever category is larger
sample = draw_sample(items, weights, 20_000)
for chosen, label in ((items, "all items"), (sample, "sample")):
    plt.hist([item.price for item in chosen], bins=range(0, 1000, 10), alpha=0.5, label=label)
plt.legend()
plt.show()

train, val, test = split(sample, 18_000, 1_000)
print(f"{len(train):,} train, {len(val):,} validation, {len(test):,} test")
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.